Prompt Strategies GraphDB

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

llm setup

In [2]:
groq_api_key = os.getenv("GROQ_API_KEY")
from langchain_groq import ChatGroq
llm = ChatGroq(api_key=groq_api_key,model_name="llama-3.1-8b-instant")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002290B637380>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002290B7C01A0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

neo4j graph setup

In [3]:
from langchain_community.graphs import Neo4jGraph
graph = Neo4jGraph(url=os.getenv("NEO4J_URI"),username=os.getenv("NEO4J_USERNAME"),password=os.getenv("NEO4J_PASSWORD"))
graph

C:\Users\Dell\AppData\Local\Temp\ipykernel_15536\720353333.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.graphs import Neo4jGraph
C:\Users\Dell\AppData\Local\Temp\ipykernel_15536\720353333.py:2: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(url=os.getenv("NEO4J_URI"),username=os.getenv("NEO4J_USERNAME"),password=os.getenv("NEO4J_PASSWORD"))


In [4]:
from langchain_neo4j import GraphCypherQAChain
chain = GraphCypherQAChain.from_llm(graph=graph,llm=llm,verbose=True,allow_dangerous_requests=True)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x000002290B7C02F0>, cypher_generation_chain=PromptTemplate(input_variables=['examples', 'question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nExamples (optional):\n{examples}\n\nThe question is:\n{question}')
| _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', '

examples

In [ ]:
examples = [
    {
        "question":"How many artists are there?",
        "query":"MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)"
    },
    {
        "question":"Which actors played in the movie Casino?",
        "query":"MATCH (m:Movie {title:'Casino'})<-[:ACTED_IN]-(a) RETURN a.name"
    },
    {
        "question":"How many movies has Tom Hanks acted in?",
        "query":"MATCH (a:Person {name:'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m)"
    },
    {
        "question":"List all the genres of the movie Schindler's List",
        "query":"MATCH (m:Movie {title: 'Schindler's List})-[:IN_GENRE]->(g:Genre) RETURN g.name"
    },
    {
        "question":"Which directors have made movies with at least three different actors named 'John'?",
        "query":"MATCH (d:Person)-[:DIRECTED]->(m:Movie)<-[ACTED_IN]-(a:Person) WHERE a.name STARTS WITH 'JOHN' WITH d, COUNT(DISTINCT a) AS JohnsCount WHERE JohnsCount >= 3 RETURN d.name"
    }
]

In [6]:
from langchain_core.prompts import FewShotPromptTemplate,PromptTemplate

example_prompt = PromptTemplate.from_template(
    "User Input: {question}\n Cypher Query: {query}"
)
prompt = FewShotPromptTemplate(
    examples = examples[:5],
    example_prompt=example_prompt,
    prefix = "You are a Neo4j expert.Given an input question, create a syntactically very accurate Cypher Query",
    suffix = "User input: {question}\n Cypher Query:",
    input_variables = ["question","schema"]
)


In [7]:
prompt

FewShotPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, examples=[{'question': 'How many artists are there?', 'query': 'MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)'}, {'question': 'Which actors played in the movie Casino?', 'query': "MATCH (m:Movie {{title:'Casino'}})<-[:ACTED_IN]-(a) RETURN a.name"}, {'question': 'How many movies has Tom Hanks acted in?', 'query': "MATCH (a:Person {{name:'Tom Hanks'}})-[:ACTED_IN]->(m:Movie) RETURN count(m)"}, {'question': "List all the genres of the movie Schindler's List", 'query': "MATCH (m:Movie {{title: 'Schindler's List}})-[:IN_GENRE]->(g:Genre) RETURN g.name"}, {'question': "Which directors have made movies with at least three different actors named 'John'?", 'query': "MATCH (d:Person)-[:DIRECTED]->(m:Movie)<-[ACTED_IN]-(a:Person) WHERE a.name STARTS WITH 'JOHN' WITH d, COUNT(DISTINCT a) AS JohnsCount WHERE JohnsCount >= 3 RETURN d.name"}], example_prompt=PromptTemplate(input_variables=['que

In [8]:
print(prompt.format(question="How many artists are there",schema="foo"))

You are a Neo4j expert.Given an input question, create a syntactically very accurate Cypher Query

User Input: How many artists are there?
 Cypher Query: MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)

User Input: Which actors played in the movie Casino?
 Cypher Query: MATCH (m:Movie {title:'Casino'})<-[:ACTED_IN]-(a) RETURN a.name

User Input: How many movies has Tom Hanks acted in?
 Cypher Query: MATCH (a:Person {name:'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m)

User Input: List all the genres of the movie Schindler's List
 Cypher Query: MATCH (m:Movie {title: 'Schindler's List})-[:IN_GENRE]->(g:Genre) RETURN g.name

User Input: Which directors have made movies with at least three different actors named 'John'?
 Cypher Query: MATCH (d:Person)-[:DIRECTED]->(m:Movie)<-[ACTED_IN]-(a:Person) WHERE a.name STARTS WITH 'JOHN' WITH d, COUNT(DISTINCT a) AS JohnsCount WHERE JohnsCount >= 3 RETURN d.name

User input: How many artists are there
 Cypher Query:


rebuilding chain w new built cypher prompt

In [10]:
chain = GraphCypherQAChain(graph=graph,llm=llm,cypher_prompt=prompt,verbose=True,allow_dangerous_requests=True,graph_schema=graph.schema)
chain

ValidationError: 2 validation errors for GraphCypherQAChain
cypher_generation_chain
  Field required [type=missing, input_value={'graph': <langchain_comm...-[:IN_GENRE]->(:Genre)'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
qa_chain
  Field required [type=missing, input_value={'graph': <langchain_comm...-[:IN_GENRE]->(:Genre)'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing